In [1]:
%load_ext autoreload
%autoreload 2

In [56]:
from bs4 import BeautifulSoup
import requests
import pandas as pd
import numpy as np
import re
from constants import KENPOM_FILE, TORVIK_FILE
from selenium import webdriver

In [3]:
def gen_browser():
    options = webdriver.FirefoxOptions()
    options.headless = True
    return webdriver.Firefox(options=options)

In [4]:
%%time
url = 'http://kenpom.com/index.php'

# Can also find archived version
# url = "https://web.archive.org/web/20220317101252/https://kenpom.com/"

# This doesn't work anymore
# f = requests.get(url)
# soup = BeautifulSoup(f.text, "lxml")

browser = gen_browser()
browser.get(url)
soup = BeautifulSoup(browser.page_source, "lxml")

CPU times: user 137 ms, sys: 13.8 ms, total: 151 ms
Wall time: 2.74 s


In [5]:
COLUMNS = [
    'Rank', 'Team', 'Conference', 'W-L', 'AdjustEM',
    'AdjustO', 'AdjustO Rank', 'AdjustD', 'AdjustD Rank',
    'AdjustT', 'AdjustT Rank', 'Luck', 'Luck Rank',
    'SOS Pyth', 'SOS Pyth Rank', 'SOS OppO', 'SOS OppO Rank',
    'SOS OppD', 'SOS OppD Rank', 'NCSOS Pyth', 'NCSOS Pyth Rank'
]

In [6]:
df = pd.DataFrame(
    [
        [tr.text for tr in row.find_all('td')]
        for row in soup.find('table', {'id': 'ratings-table'}).find_all(class_=r"tourney")
    ],
    columns=COLUMNS
)

In [7]:
df.shape

(68, 21)

In [8]:
# Returns true if given string is a number and a valid seed number (1-16)
def is_valid_seed(x):
    return str(x).replace(' ', '').isdigit() and int(x) > 0 and int(x) <= 16


def parse_name(row):
    if is_valid_seed(row['Team'][-2:]):
        row['Seed'] = row['Team'][-2:].strip()
        row['Team'] = row['Team'][:-2].strip()
    else:
        row['Seed'] = np.NaN
        row['Team'] = row['Team']
    return row


# Parse out seed/team
df = df.apply(parse_name, axis=1)

In [9]:
# Split W-L column into wins and losses
df['Wins'] = df['W-L'].apply(lambda x: int(re.sub('-.*', '', x)))
df['Losses'] = df['W-L'].apply(lambda x: int(re.sub('.*-', '', x)))
df.drop('W-L', inplace=True, axis=1)

# Reorder columns just cause I'm OCD
df = df[['Rank', 'Seed', 'Team', 'Conference', 'Wins', 'Losses', 'AdjustEM',
         'AdjustO', 'AdjustO Rank', 'AdjustD', 'AdjustD Rank',
         'AdjustT', 'AdjustT Rank', 'Luck', 'Luck Rank',
         'SOS Pyth', 'SOS Pyth Rank', 'SOS OppO', 'SOS OppO Rank',
         'SOS OppD', 'SOS OppD Rank', 'NCSOS Pyth', 'NCSOS Pyth Rank']]

# Drop non tournament teams
df = df.dropna()

In [10]:
df

,Rank,Seed,Team,Conference,Wins,Losses,AdjustEM,AdjustO,AdjustO Rank,AdjustD,...,Luck,Luck Rank,SOS Pyth,SOS Pyth Rank,SOS OppO,SOS OppO Rank,SOS OppD,SOS OppD Rank,NCSOS Pyth,NCSOS Pyth Rank
0,1,1,Duke,ACC,32,2,+38.90,128.0,4,89.1,...,+.049,63,+14.29,15,117.0,24,102.7,10,+10.07,18
1,2,1,Arizona,B12,32,2,+37.66,127.7,5,90.0,...,+.023,127,+14.97,9,117.6,15,102.7,7,+3.25,102
2,3,1,Michigan,B10,31,3,+37.59,126.6,8,89.0,...,+.045,70,+16.65,3,119.1,4,102.5,5,+12.49,11
3,4,1,Florida,SEC,26,7,+33.79,125.5,9,91.7,...,-.036,289,+16.01,5,119.4,3,103.4,26,+7.96,29
4,5,2,Houston,B12,28,6,+33.43,124.9,14,91.4,...,-.006,203,+13.58,24,116.9,25,103.3,24,+0.87,156
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
63,192,16,Siena,MAAC,23,11,-2.10,107.1,210,109.2,...,+.005,177,-9.48,347,102.9,350,112.4,297,-7.72,343
64,207,16,Howard,MEAC,23,10,-3.19,103.1,283,106.3,...,-.010,210,-14.60,364,100.5,363,115.1,361,-1.69,234
65,216,16,LIU,NEC,24,10,-3.95,105.6,239,109.6,...,+.104,12,-9.97,350,103.6,334,113.5,343,+2.14,119
66,284,16,Lehigh,PL,18,16,-10.37,102.7,290,113.1,...,+.081,23,-8.61,330,104.9,304,113.5,342,-0.54,199


In [11]:
df.to_csv(KENPOM_FILE.format(gender="mens"), index=False)

## Scrape Torvik

In [28]:
YEAR = 2026
url = f'https://barttorvik.com/ncaaw/trank.php?year={YEAR}&conlimit=NCAA'

In [30]:
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.common.by import By

browser = gen_browser()
browser.get(url)
element = WebDriverWait(browser, 10).until(
    EC.visibility_of_element_located((By.CLASS_NAME, "seedrow"))
)
soup = BeautifulSoup(browser.page_source, "lxml")

In [59]:
df = pd.DataFrame(
    [
        [tr.text for tr in row.find_all('td')]
        for row in soup.find_all(class_="seedrow")
    ],
).iloc[:, [1, 7]].rename(columns={7: 'pyth'})
df['Team'] = df[1].str.split("\xa0\xa0\xa0").apply(lambda x: x[0])
df['Seed'] = df[1].str.split("\xa0\xa0\xa0").apply(lambda x: x[1]).str.split(" seed").apply(lambda x: x[0])
df[['Team', 'Seed', 'pyth']].to_csv(TORVIK_FILE.format(gender='womens'), index=False)

# Old Version
This doesn't work with NIT seeds

In [ ]:
table_html = soup.find_all('table', {'id': 'ratings-table'})

In [3]:
# Weird issue w/ <thead> in the html
# Prevents us from just using pd.read_html
# Let's find all the thead contents and just replace/remove them
# This allows us to easily put the table row data into a dataframe using pandas
thead = table_html[0].find_all('thead')

table = table_html[0]
for x in thead:
    table = str(table).replace(str(x), '')

In [4]:
#    table = "<table id='ratings-table'>%s</table>" % table
df = pd.read_html(table, converters={3: lambda x: str(x)})[0]

df.columns = COLUMNS